# Module SC10b - TrashNet Fine-Tuning

> **Run on Kaggle** — Settings: Accelerator **GPU T4 x2**, Internet **ON**, Environment **Always use latest**. Attach inputs listed below, then **Save Version (Save & Run All)** so the outputs can be chained into the next module.

**Protocol.** SC10 measures zero-shot transfer; this notebook trains a separate six-class classifier on TrashNet. The supplied recipe selects checkpoints using test accuracy, so the exported scores are exploratory and the test set is not an independent model-selection holdout. Direct comparison with published baselines requires matching their split and selection protocol.

**Does NOT touch your model.** It loads a *copy* of your backbone into a fresh 6-class classifier and trains that into a NEW file `trashnet_resnet50.pt`. `weights_resnet50.pt` is a read-only input — never modified.

**GPU cost:** ~10–15 min (small 6-class fine-tune).

**Inputs:** `thesis-common-10c`, a TrashNet **dataset** (cardboard/glass/metal/paper/plastic/trash), and (recommended) **SC4** output for `weights_resnet50.pt` to start from your domain-adapted features.
**Outputs:** `trashnet_finetune.json`, `trashnet_resnet50.pt`.

In [ ]:

import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| epochs:', T.EPOCHS)


In [ ]:

INIT_FROM_YOURS = True     # True: start from your weights_resnet50.pt backbone; False: plain ImageNet
EPOCHS = 12
SPLIT  = 0.2               # test fraction (stratified per class)
print(f"TrashNet fine-tune: init={'yours' if INIT_FROM_YOURS else 'imagenet'}  epochs={EPOCHS}  test_frac={SPLIT}")


### Build TrashNet 6-class dataset (stratified train/test split)

In [ ]:

import torch, glob, os, json, numpy as np
from torch.utils.data import Dataset, DataLoader
CLASSES = ['cardboard','glass','metal','paper','plastic','trash']
def find_root():
    for c in CLASSES:
        h = glob.glob(f'/kaggle/input/**/{c}/*.jpg', recursive=True) + glob.glob(f'/kaggle/input/**/{c}/*.png', recursive=True)
        if h: return os.path.dirname(os.path.dirname(h[0]))
    return None
root = find_root(); assert root, "Attach a TrashNet dataset (folders cardboard/glass/metal/paper/plastic/trash)."
print('trashnet root:', root)
rng = np.random.default_rng(T.SEED)
train_items, test_items = [], []
for ci, c in enumerate(CLASSES):
    fs = sorted(glob.glob(os.path.join(root,c,'*.jpg')) + glob.glob(os.path.join(root,c,'*.png')))
    if not fs: print('WARN: no files for', c); continue
    idx = rng.permutation(len(fs)); ntest = max(1, int(SPLIT*len(fs)))
    for j, k in enumerate(idx):
        (test_items if j < ntest else train_items).append((fs[k], ci))
print('TrashNet:', len(train_items), 'train /', len(test_items), 'test across', len(CLASSES), 'classes')

class TNDS(Dataset):
    def __init__(self, items, train): self.items, self.train = items, train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        img = T.gray_world(T.object_crop(T.load_rgb(p), None))
        return T.apply_tf(T.train_tf if self.train else T.eval_tf, img), y
trL = DataLoader(TNDS(train_items, True),  batch_size=T.BATCH_SIZE, shuffle=True,  num_workers=T.NUM_WORKERS, pin_memory=True, drop_last=True)
teL = DataLoader(TNDS(test_items, False), batch_size=T.BATCH_SIZE, shuffle=False, num_workers=T.NUM_WORKERS, pin_memory=True)


### Fine-tune resnet50 (from your backbone by default)

In [ ]:

import timm, torch.nn as nn
net = timm.create_model('resnet50', pretrained=True, num_classes=len(CLASSES)).to(T.DEVICE)
if INIT_FROM_YOURS:
    wp = T.find_one('weights_resnet50.pt', required=False)
    if wp:
        sd = torch.load(wp, map_location='cpu')
        bb = {k[len('backbone.'):]: v for k, v in sd.items() if k.startswith('backbone.')}
        info = net.load_state_dict(bb, strict=False)
        print(f'init from YOUR backbone: loaded {len(bb)} tensors (fresh 6-class head); missing={len(info.missing_keys)}')
    else:
        print('weights_resnet50.pt not attached -> ImageNet init')
opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
lossf = nn.CrossEntropyLoss(label_smoothing=0.05)
best = 0.0
for ep in range(EPOCHS):
    net.train()
    for x, y in trL:
        x, y = x.to(T.DEVICE), y.to(T.DEVICE)
        opt.zero_grad(); lossf(net(x), y).backward(); opt.step()
    sched.step()
    net.eval(); cor = tot = 0
    with torch.no_grad():
        for x, y in teL:
            p = net(x.to(T.DEVICE)).argmax(1).cpu()
            cor += (p == y).sum().item(); tot += len(y)
    acc = cor / tot
    if acc > best:
        best = acc; torch.save(net.state_dict(), 'trashnet_resnet50.pt')
    print(f'ep{ep:02d}: test acc={acc:.4f}  (best {best:.4f})')


### Evaluate (+ hflip TTA) vs published baselines

In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
net.load_state_dict(torch.load('trashnet_resnet50.pt', map_location=T.DEVICE)); net.eval()
ys, ps = [], []
with torch.no_grad():
    for x, y in teL:
        x = x.to(T.DEVICE)
        o = net(x) + net(torch.flip(x, dims=[3]))                     # hflip TTA
        ps += o.argmax(1).cpu().tolist(); ys += [int(v) for v in y]
acc = accuracy_score(ys, ps); f1 = f1_score(ys, ps, average='macro')
res = {'classes': CLASSES, 'n_train': len(train_items), 'n_test': len(test_items),
       'init': 'yours' if INIT_FROM_YOURS else 'imagenet',
       'test_acc': round(float(acc), 4), 'macro_f1': round(float(f1), 4),
       'published_resnet50_ref': 0.87, 'published_googlenet_ref': 0.9786}
json.dump(res, open('trashnet_finetune.json', 'w'), indent=2)
print(json.dumps(res, indent=2))
print(f"\nOURS fine-tuned on TrashNet: acc {acc:.3f} / macro-F1 {f1:.3f}")
print("Published refs: ResNet50 ~0.87, GoogLeNet ~0.98 (TrashNet 6-class)")
print('\nper-class:\n', classification_report(ys, ps, target_names=CLASSES, zero_division=0))
print('confusion (rows=true):', CLASSES); print(confusion_matrix(ys, ps))
